In [ ]:
Surg_BM
Surg_Bl
Surg_Hip
PM_Ctx
PM_Bl


In [ ]:
import snapatac2 as snap
import os
import sys
import scanpy as sc
import pandas as pd 
import anndata as ad
import scvi


/home/elliot/miniconda3/envs/atac_env190516/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
Surg_BM = snap.pp.import_fragments(
    "/mnt/claw-raid/elliot/P002_mtscATAC-seq/data/raw/bm_epilepsy/outs/fragments.tsv.gz",
    chrom_sizes=snap.genome.hg38,
    sorted_by_barcode=False,
)

In [23]:
Surg_BM.obs

,n_fragment,frac_dup,frac_mito,tsse
AAACGAAAGACCCTAT-1,362,0.457271,0.0,1.010101
AAACGAAAGACGTCAG-1,203,0.363636,0.0,0.000000
AAACGAAAGAGGCAGG-1,274,0.459566,0.0,0.790514
AAACGAAAGCACGTAG-1,224,0.348837,0.0,0.000000
AAACGAAAGCAGAGCT-1,240,0.398496,0.0,0.000000
...,...,...,...,...
TTTGTGTTCTATCTTG-1,371,0.403537,0.0,1.346801
TTTGTGTTCTCCAACC-1,25603,0.310840,0.0,13.980369
TTTGTGTTCTCGCGTT-1,282,0.440476,0.0,0.673401
TTTGTGTTCTGAAAGA-1,219,0.368876,0.0,0.865801


In [24]:
import plotly.graph_objects as go

fig = go.Figure()

for adata, name, colour in [
    (Pm_Bl, "Pm_Bl", "#76C3A9"),
    (Surg_Bl, "Surg_Bl", "#355C4C"),
    (Surg_BM, "Surg_BM", "#7EA040")
]:
    fig.add_trace(go.Violin(
        y=adata.obs["tsse"].dropna(),
        name=name,
        box_visible=True,
        points=False,
        fillcolor=colour,
        line_color="black",
        opacity=0.7
    ))

fig.update_layout(
    template="simple_white",
    yaxis_title="TSS enrichment",
    xaxis_title="",
    showlegend=False,
    width=500,
    height=500
)

fig.show()

In [ ]:
#Pm_Ctx and Surg_Ctx need processing 

In [12]:
Pm_Bl = ad.read_h5ad('/mnt/claw-raid/elliot/P002_mtscATAC-seq/data/processed/atac_PM_blood/anndata_PBMC_PM.h5ad')

In [ ]:
#source code for TSSE plot  
import plotly.graph_objects as go

    selected_cells = adata.obs["n_fragment"] >= min_fragment
    x = adata.obs["n_fragment"][selected_cells]
    y = adata.obs["tsse"][selected_cells]

    log_x = np.log10(x)
    log_x_min, log_x_max = log_x.min(), log_x.max()
    x_range = log_x_max - log_x_min
    bin_x = np.linspace(log_x_min - 0.1 * x_range, log_x_max + 0.2 * x_range, 20)

    y_min, y_max = y.min(), y.max()
    y_range = y_max - y_min
    bin_y = np.linspace(y_min - 0.1 * y_range, y_max + 0.2 * y_range, 15)

    (z, rx, ry) = np.histogram2d(log_x,y, bins=(bin_x, bin_y))
    fig = go.Figure(data =
        go.Contour(
            z = z.T,
            x = 10**rx,
            y = ry,
            colorscale = 'Blues',
        )
    )

    if show_cells:
        fig.add_trace(go.Scatter(
                x = x,
                y = y,
                xaxis = 'x',
                yaxis = 'y',
                mode = 'markers',
                marker = dict(color = 'rgba(0,0,0,0.3)', size = 3)
        ))

    fig.update_xaxes(type="log")
    fig.update_layout(
        template="simple_white",
        xaxis_title="Number of unique fragments",
        yaxis_title="TSS enrichment score",
    )

    return render_plot(fig, width, height, interactive, show, out_file)


#cryo mt sc-atac code
qc_NB <- data.frame(Sample = NB$Sample, TSS_enrichment = NB$TSS.enrichment)
qc_GN <- data.frame(Sample = "GN01", TSS_enrichment = GN$TSS.enrichment)
qc_combined <- rbind(qc_NB, qc_GN)
qc_combined$Sample <- factor(qc_combined$Sample, levels = c(unique(seurat$Sample), "GN01"))

ggplot(qc_combined, aes(x = Sample, y = TSS_enrichment, fill = Sample)) +
  geom_violin(trim = FALSE, scale = "width", width = 0.9, alpha = 0.3) +
  geom_boxplot(width = 0.4, outlier.shape = NA, color = "black", alpha = 0.8) +
  scale_fill_manual(values = c("#76C3A9", "#355C4C", "#b37bb3")) +
  labs(y = "TSS enrichment", x = "") +       # Note: previously "nCount ATAC", should reflect TSS
  theme_classic() + ylim(0, 10) +
  theme(text = element_text(colour = "black", size = 15),
        axis.line = element_line(color = "black", size = 0.8),
        axis.text.x = element_text(colour = "black", size = 15, hjust = 1),
        axis.text.y = element_text(colour = "black", size = 15),
        axis.ticks.y = element_line(color = "black", size = 0.8),
        axis.ticks.x = element_line(color = "black", size = 0))